# 🤖 AI Functions Showcase - The Art of the Possible

**COMPREHENSIVE AI DEMONSTRATION** - Run this after notebook 01.

## What this notebook demonstrates:
- ✅ **ai_classify** - Intelligent priority and category classification
- ✅ **ai_extract** - Structured data extraction using entity labels
- ✅ **ai_gen** - Complex analysis, summaries, and creative content generation
- ✅ **Final Summary Table** - Complete AI-powered ticket analysis

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Showcase:
- `ai_classify` - For priority and category classification
- `ai_extract` - For structured data extraction using entity labels
- `ai_gen` - For complex analysis and content generation

---

## 🎯 Goal: Demonstrate the full power of Databricks AI Functions


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🚀 Ready to showcase Databricks AI Functions!")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
🚀 Ready to showcase Databricks AI Functions!


In [2]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample ticket data:")
display(df_tickets.select("ticket_id", "short_description", "description").limit(3))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample ticket data:


,ticket_id,short_description,description
0,TICKET_001,Issue #1 - Critical,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.
1,TICKET_002,Issue #2 - Important,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.
2,TICKET_003,Issue #3 - Problem,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?


# 🎯 AI Function #1: ai_classify

**Purpose:** Intelligent classification of text into predefined categories

**Use Cases:** Priority classification, category assignment, sentiment analysis, risk assessment

**Example:** Classify ticket priorities based on description content


In [3]:
# AI Function #1: ai_classify - Priority Classification
print("🎯 Demonstrating ai_classify for priority classification...")

# Register DataFrame as temporary view for SQL access
df_tickets.createOrReplaceTempView("tickets")

# Use ai_classify to classify ticket priorities
df_with_priority = spark.sql("""
    SELECT
        *,
        ai_classify(
            description, 
            ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')
        ) as ai_priority_classification
    FROM tickets
""")

print("✅ ai_classify completed - Priority classification done!")
print("\n📊 Priority Classification Results:")
display(df_with_priority.select("ticket_id", "short_description", "ai_priority_classification").limit(5))

# Show distribution of AI classifications
print("\n📈 Priority Distribution:")
df_with_priority.groupBy("ai_priority_classification").count().orderBy(desc("count")).show()


🎯 Demonstrating ai_classify for priority classification...
✅ ai_classify completed - Priority classification done!

📊 Priority Classification Results:


,ticket_id,short_description,ai_priority_classification
0,TICKET_001,Issue #1 - Critical,Urgent Priority
1,TICKET_002,Issue #2 - Important,Urgent Priority
2,TICKET_003,Issue #3 - Problem,Medium Priority
3,TICKET_004,Issue #4 - Critical,High Priority
4,TICKET_005,Issue #5 - Problem,High Priority



📈 Priority Distribution:


+--------------------------+-----+
|ai_priority_classification|count|
+--------------------------+-----+
|           Urgent Priority|    6|
|             High Priority|    3|
|           Medium Priority|    1|
+--------------------------+-----+



# 🎯 AI Function #2: ai_extract

**Purpose:** Extract structured data from unstructured text using entity labels

**Use Cases:** Action items extraction, requirement identification, urgency assessment, system identification

**Example:** Extract specific entities (action_items, main_requirement, urgency_level, affected_systems) from ticket descriptions


In [4]:
# AI Function #2: ai_extract - Structured Data Extraction
print("🎯 Demonstrating ai_extract for structured data extraction...")

# Register DataFrame as temporary view for SQL access
df_with_priority.createOrReplaceTempView("tickets_with_priority")

# Use ai_extract with correct ARRAY<STRING> syntax for labels
df_with_extraction = spark.sql("""
    SELECT
        *,
        ai_extract(
            description,
            ARRAY('action_items', 'main_requirement', 'urgency_level', 'affected_systems')
        ) as ai_extracted_data
    FROM tickets_with_priority
""")

print("✅ ai_extract completed - Structured data extraction done!")
print("\n📊 Extraction Results:")
display(df_with_extraction.select("ticket_id", "short_description", "ai_extracted_data").limit(3))

# Parse the extracted data for better display
df_parsed = df_with_extraction.withColumn(
    "action_items", 
    col("ai_extracted_data.action_items")
).withColumn(
    "main_requirement", 
    col("ai_extracted_data.main_requirement")
).withColumn(
    "urgency_level", 
    col("ai_extracted_data.urgency_level")
).withColumn(
    "affected_systems", 
    col("ai_extracted_data.affected_systems")
)

print("\n📋 Parsed Extraction Results:")
display(df_parsed.select("ticket_id", "action_items", "main_requirement", "urgency_level").limit(3))


🎯 Demonstrating ai_extract for structured data extraction...
✅ ai_extract completed - Structured data extraction done!

📊 Extraction Results:


,ticket_id,short_description,ai_extracted_data
0,TICKET_001,Issue #1 - Critical,"{'action_items': 'can someone look into this', 'main_requirement': 'our website is super slow today and customers are complaining', 'urgency_level': 'we're losing sales', 'affected_systems': 'our website, the database'}"
1,TICKET_002,Issue #2 - Important,"{'action_items': 'fix this', 'main_requirement': 'fix the login system', 'urgency_level': 'urgent', 'affected_systems': 'login system'}"
2,TICKET_003,Issue #3 - Problem,"{'action_items': 'debug this', 'main_requirement': 'help with the new feature', 'urgency_level': None, 'affected_systems': 'the API'}"



📋 Parsed Extraction Results:


,ticket_id,action_items,main_requirement,urgency_level
0,TICKET_001,can someone look into this?,our website is super slow today and customers are complaining,we're losing sales
1,TICKET_002,fix this,fix the login system,urgent
2,TICKET_003,debug this,help with the new feature,None


# 🎯 AI Function #3: ai_gen

**Purpose:** Generate creative content, summaries, and complex analysis

**Use Cases:** Executive summaries, detailed analysis, creative content, recommendations

**Example:** Generate comprehensive ticket analysis and recommendations


In [5]:
# AI Function #3: ai_gen - Comprehensive Analysis
print("🎯 Demonstrating ai_gen for comprehensive analysis...")

# Register DataFrame as temporary view for SQL access
df_parsed.createOrReplaceTempView("tickets_with_extraction")

# Use ai_gen to create comprehensive analysis
df_with_analysis = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Analyze this IT ticket and provide: ',
                '1. Executive summary (2-3 sentences), ',
                '2. Technical complexity (Low/Medium/High), ',
                '3. Estimated effort (hours), ',
                '4. Risk assessment (Low/Medium/High), ',
                '5. Recommended next steps. ',
                'Ticket: ', short_description, ' - ', description
            )
        ) as ai_comprehensive_analysis
    FROM tickets_with_extraction
""")

print("✅ ai_gen completed - Comprehensive analysis done!")
print("\n📊 Analysis Results:")
display(df_with_analysis.select("ticket_id", "short_description", "ai_comprehensive_analysis").limit(3))


🎯 Demonstrating ai_gen for comprehensive analysis...
✅ ai_gen completed - Comprehensive analysis done!

📊 Analysis Results:


,ticket_id,short_description,ai_comprehensive_analysis
0,TICKET_001,Issue #1 - Critical,"Here's the analysis of the IT ticket:\n\n**1. Executive Summary**: The company's website is experiencing critical performance issues, resulting in slow load times and lost sales. The issue has been ongoing since morning, and customers are actively complaining. The root cause is suspected to be related to the database, but requires further investigation.\n\n**2. Technical Complexity**: Medium - The issue is likely related to the database or website infrastructure, which may involve multiple components and potential causes, requiring some technical expertise to diagnose and resolve.\n\n**3. Estimated Effort**: 4-6 hours - Depending on the root cause, resolving the issue may require a few hours of investigation, troubleshooting, and potential fixes, such as optimizing database queries, checking server resources, or updating website configurations.\n\n**4. Risk Assessment**: High - The issue is critical, and the ongoing performance problems are directly impacting sales and customer satisfaction, which can lead to reputational damage and financial losses if not resolved promptly.\n\n**5. Recommended Next Steps**: \n* Immediately assign a technical resource to investigate the issue and gather more information about the website's performance, such as error logs, server metrics, and database performance metrics.\n* Conduct a preliminary analysis to identify potential bottlenecks or issues, such as database queries, server resource utilization, or network connectivity problems.\n* Collaborate with the development team and other stakeholders to review recent changes or updates that may be contributing to the issue.\n* Develop a plan to implement temporary fixes or optimizations to improve website performance while a more permanent solution is being worked on."
1,TICKET_002,Issue #2 - Important,"Here's the analysis of the IT ticket:\n\n1. **Executive Summary**: The login system is down, preventing users from accessing the system and causing a high volume of support calls. This is a recurring issue that occurred last week, and prompt resolution is necessary to prevent further customer dissatisfaction and potential loss. The situation requires immediate attention to minimize business impact.\n2. **Technical Complexity**: Medium - The issue is likely related to a specific component or configuration of the login system, but the fact that it's a recurring problem suggests that the root cause may be more complex and require some investigation to resolve.\n3. **Estimated Effort**: 4-6 hours - Assuming the issue is similar to the one that occurred last week, the effort required to resolve it may involve some troubleshooting, analysis, and potential code or configuration changes. However, if the root cause is more complex, the effort required may be higher.\n4. **Risk Assessment**: High - The issue is already causing significant disruption to business operations, and the longer it takes to resolve, the higher the risk of customer dissatisfaction, loss of business, and damage to the company's reputation.\n5. **Recommended Next Steps**: \n* Immediately assemble a team to investigate the issue and identify the root cause.\n* Review the incident from last week to see if there are any similarities or clues that can help resolve the current issue.\n* Prioritize troubleshooting and analysis to quickly identify the cause of the problem.\n* Develop a plan to implement a fix, including any necessary code or configuration changes, and test it thoroughly to ensure the issue is fully resolved.\n* Consider implementing temporary measures to mitigate the impact on customers, such as providing alternative access methods or communicating the status of the issue through social media or email updates."
2,TICKET_003,Issue #3 - Problem,"Here's the analysis of the IT ticket:\n\n1. **Executive Summary**: The developer is experiencing intermittent issues with a new feature's API, wh

# 🎉 Final Summary Table - The Art of the Possible

**Complete AI-powered ticket analysis showcasing all three AI functions**


In [6]:
# Create Final Summary Table
print("🎉 Creating comprehensive summary table...")

# Create a clean summary table with all AI results
df_final_summary = df_with_analysis.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "action_items",
    "main_requirement", 
    "urgency_level",
    "affected_systems",
    "ai_comprehensive_analysis"
).withColumn(
    "ai_showcase_timestamp", 
    current_timestamp()
)

print("✅ Final summary table created!")
print(f"📊 Total tickets analyzed: {df_final_summary.count()}")

# Display the comprehensive results
print("\n🎯 COMPLETE AI SHOWCASE RESULTS:")
print("="*80)
display(df_final_summary.limit(5))

# Save to Unity Catalog
print("\n💾 Saving results to Unity Catalog...")
df_final_summary.write.format("delta").mode("overwrite").saveAsTable(TABLES["ai_showcase_results"])
print(f"✅ Results saved to: {TABLES['ai_showcase_results']}")

# Show summary statistics
print("\n📈 AI Showcase Summary Statistics:")
print(f"🎯 Tickets processed: {df_final_summary.count()}")
print(f"🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)")
print(f"📊 Data saved to Unity Catalog: {TABLES['ai_showcase_results']}")

print("\n" + "="*80)
print("🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!")
print("="*80)
print("✅ ai_classify: Priority classification")
print("✅ ai_extract: Structured data extraction using entity labels")
print("✅ ai_gen: Comprehensive analysis and content generation")
print("✅ Final summary table created and saved")
print("="*80)


🎉 Creating comprehensive summary table...
✅ Final summary table created!


📊 Total tickets analyzed: 10

🎯 COMPLETE AI SHOWCASE RESULTS:


,ticket_id,short_description,ai_priority_classification,action_items,main_requirement,urgency_level,affected_systems,ai_comprehensive_analysis,ai_showcase_timestamp
0,TICKET_001,Issue #1 - Critical,Urgent Priority,can someone look into this?,our website is super slow today,we're losing sales,our website,"Here's the analysis of the IT ticket:\n\n**1. Executive Summary**: The company's website is experiencing critical performance issues, resulting in slow load times and lost sales. The issue has been ongoing since morning, and customers are actively complaining. The root cause is suspected to be related to the database, but requires further investigation.\n\n**2. Technical Complexity**: Medium - The issue is likely related to a technical component such as the database, but the exact cause is unknown, requiring some troubleshooting and analysis to identify the root cause.\n\n**3. Estimated Effort**: 4-6 hours - This estimate assumes that the issue can be identified and resolved relatively quickly, but may require some time to troubleshoot, analyze logs, and potentially optimize database performance or resolve other technical issues.\n\n**4. Risk Assessment**: High - The issue is critical, and the company is already experiencing lost sales and customer complaints, which can damage the company's reputation and revenue. The longer the issue persists, the higher the risk of further financial losses and reputational damage.\n\n**5. Recommended Next Steps**: \n* Immediately assign a technical resource to investigate the issue and gather more information about the website's performance, such as error logs and system metrics.\n* Check the database and server performance to identify any bottlenecks or issues.\n* Consider engaging a database administrator or a web developer to assist with troubleshooting and resolution.\n* Provide regular updates to stakeholders on the progress of the investigation and estimated time to resolution.\n* Develop a plan to mitigate the issue and prevent similar occurrences in the future, such as implementing monitoring and alerting tools to detect performance issues earlier.",2025-09-20 01:09:20.876165
1,TICKET_002,Issue #2 - Important,Urgent Priority,we need to fix this asap,fix the login system,high,login system,"Here's the analysis of the IT ticket:\n\n1. **Executive Summary**: The login system is down, preventing users from accessing the system and causing a high volume of support calls. This is a recurring issue that occurred last week, and prompt resolution is necessary to prevent further customer dissatisfaction and potential loss. The situation requires immediate attention to minimize business impact.\n2. **Technical Complexity**: Medium - The issue is likely related to a specific component or configuration of the login system, but the fact that it's a recurring problem suggests that the root cause may be more complex and require some investigation to resolve.\n3. **Estimated Effort**: 4-6 hours - Assuming the issue is similar to the one that occurred last week, the effort required to resolve it may involve reviewing previous fixes, troubleshooting the current issue, and implementing a more permanent solution to prevent future recurrences.\n4. **Risk Assessment**: High - The login system is a critical component of the business, and its failure is causing significant disruption to users and support operations. If not resolved quickly, the issue may lead to a loss of customer trust and revenue.\n5. **Recommended Next Steps**: \n * Immediately assemble a team to investigate the issue and review previous fixes.\n * Conduct a thorough analysis of the login system's logs and configuration to identify the root cause.\n * Implement a temporary fix to restore login functionality as soon as possible.\n * Schedule a follow-up task to review the login system's architecture and implement a more permanent solution to prevent future recurrences.",2025-09-20 01:09:20.876165
2,TICKET_003,Issue #3 - Problem,Medium Prior


💾 Saving results to Unity Catalog...


✅ Results saved to: quickstart_catalog_vkm_external.classify_tickets.ai_showcase_results

📈 AI Showcase Summary Statistics:


🎯 Tickets processed: 10
🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)
📊 Data saved to Unity Catalog: quickstart_catalog_vkm_external.classify_tickets.ai_showcase_results

🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!
✅ ai_classify: Priority classification
✅ ai_extract: Structured data extraction using entity labels
✅ ai_gen: Comprehensive analysis and content generation
✅ Final summary table created and saved
